# NB01 — Canonical Residual Training and K562 Tracking

This notebook trains the **canonical control-anchored residual model** and compares:

1. residual-only
2. residual + cell-aware MMD

**What to send back after running**
- epoch logs
- best checkpoint summary
- final overall metrics
- final per-cell metrics
- especially K562 test/OOD


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install anndata scanpy torch pandas scikit-learn scipy

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 59.7 MB/s eta 0:00:00


In [ ]:
import os, random, pickle
from dataclasses import dataclass

import anndata as ad
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score
from scipy.stats import pearsonr

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

@dataclass
class CFG:
    data_path: str = "/content/drive/MyDrive/ChemDFM/data/sciplex_complete_middle_subset.h5ad"
    results_dir: str = "/content/drive/MyDrive/ChemDFM/results_nb01"
    ckpt_dir: str = "/content/drive/MyDrive/ChemDFM/checkpoints_nb01"
    split_col: str = "split_ho_pathway"
    pca_dim: int = 25
    batch_size: int = 512
    epochs: int = 12
    lr: float = 1e-3
    wd: float = 1e-5
    emb_dim: int = 32
    hidden: int = 256
    dose_hidden: int = 32
    alpha_topk: float = 2.0
    alpha_mmd: float = 0.05
    dropout: float = 0.1
    pin_memory: bool = False
    num_workers: int = 0
    condition_col: str = "condition"
    cell_col: str = "cell_type"
    dose_col: str = "dose"
    fallback_dose_col: str = "dose_val"
cfg = CFG()
os.makedirs(cfg.results_dir, exist_ok=True)
os.makedirs(cfg.ckpt_dir, exist_ok=True)

Using device: cpu


In [ ]:
adata = ad.read_h5ad(cfg.data_path)
if cfg.dose_col not in adata.obs.columns and cfg.fallback_dose_col in adata.obs.columns:
    adata.obs[cfg.dose_col] = adata.obs[cfg.fallback_dose_col]

def map_split(x: str) -> str:
    x = str(x).lower()
    if "train" in x:
        return "train"
    if "test" in x or "val" in x:
        return "test"
    if "ood" in x:
        return "ood"
    return "drop"

adata.obs["_split3"] = adata.obs[cfg.split_col].astype(str).map(map_split)
adata = adata[adata.obs["_split3"].isin(["train","test","ood"])].copy()

X = adata.X
if hasattr(X, "toarray"):
    X = X.toarray()
X = np.asarray(X, dtype=np.float32)

train_mask = (adata.obs["_split3"] == "train").values
pca = PCA(n_components=cfg.pca_dim, random_state=SEED)
X_pca = np.zeros((adata.n_obs, cfg.pca_dim), dtype=np.float32)
X_pca[train_mask] = pca.fit_transform(X[train_mask]).astype(np.float32)
X_pca[~train_mask] = pca.transform(X[~train_mask]).astype(np.float32)

with open(f"{cfg.results_dir}/pca_model.pkl", "wb") as f:
    pickle.dump(pca, f)

drug_enc = LabelEncoder()
cell_enc = LabelEncoder()
drug_enc.fit(adata.obs[cfg.condition_col].astype(str))
cell_enc.fit(adata.obs[cfg.cell_col].astype(str))
adata.obs["drug_idx"] = drug_enc.transform(adata.obs[cfg.condition_col].astype(str))
adata.obs["cell_idx"] = cell_enc.transform(adata.obs[cfg.cell_col].astype(str))
idx2cell = {i:c for i,c in enumerate(cell_enc.classes_)}
print("Cell types:", idx2cell)

control_mask = (adata.obs[cfg.condition_col].astype(str) == "control").values
ctrl_means = {}
for cell in sorted(adata.obs[cfg.cell_col].astype(str).unique()):
    m = control_mask & (adata.obs[cfg.cell_col].astype(str).values == cell)
    ctrl_means[cell] = X_pca[m].mean(axis=0).astype(np.float32)

X0_pca = np.stack([ctrl_means[c] for c in adata.obs[cfg.cell_col].astype(str).values]).astype(np.float32)
DELTA_pca = (X_pca - X0_pca).astype(np.float32)

Cell types: {0: 'A549', 1: 'K562', 2: 'MCF7'}


In [ ]:
class ChemResidualDataset(Dataset):
    def __init__(self, split):
        mask = (adata.obs["_split3"].values == split) & (adata.obs[cfg.condition_col].astype(str).values != "control")
        self.idxs = np.where(mask)[0]
    def __len__(self):
        return len(self.idxs)
    def __getitem__(self, i):
        idx = self.idxs[i]
        row = adata.obs.iloc[idx]
        dose = float(row[cfg.dose_col])
        dose = np.log1p(max(dose, 0.0))
        return {
            "x_true": torch.tensor(X_pca[idx], dtype=torch.float32),
            "x0": torch.tensor(X0_pca[idx], dtype=torch.float32),
            "delta": torch.tensor(DELTA_pca[idx], dtype=torch.float32),
            "drug_idx": torch.tensor(int(row["drug_idx"]), dtype=torch.long),
            "cell_idx": torch.tensor(int(row["cell_idx"]), dtype=torch.long),
            "dose": torch.tensor([dose], dtype=torch.float32),
            "condition": str(row[cfg.condition_col]),
            "cell_type": str(row[cfg.cell_col]),
        }

train_ds = ChemResidualDataset("train")
test_ds = ChemResidualDataset("test")
ood_ds = ChemResidualDataset("ood")

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers, pin_memory=cfg.pin_memory)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=cfg.pin_memory)
ood_loader = DataLoader(ood_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=cfg.pin_memory)

print("Train/Test/OOD perturbed:", len(train_ds), len(test_ds), len(ood_ds))

Train/Test/OOD perturbed: 281243 49834 10559


In [ ]:
class MLP(nn.Module):
    def __init__(self, in_dim, hidden_dims, out_dim, dropout=0.1):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, out_dim))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

class StructuredDoseEncoder(nn.Module):
    def __init__(self, out_dim=32):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(1,16), nn.ReLU(), nn.Linear(16,out_dim))
    def forward(self, dose):
        return self.net(dose)

class ResidualDoseResponseModel(nn.Module):
    def __init__(self, latent_dim, n_drugs, n_cells, emb_dim=32, hidden=256, dose_hidden=32, dropout=0.1):
        super().__init__()
        self.drug_emb = nn.Embedding(n_drugs, emb_dim)
        self.cell_emb = nn.Embedding(n_cells, emb_dim)
        self.dose_enc = StructuredDoseEncoder(dose_hidden)
        self.ctrl_enc = MLP(latent_dim, [hidden, hidden], hidden, dropout)
        fusion_in = hidden + emb_dim + emb_dim + dose_hidden
        self.delta_head = MLP(fusion_in, [hidden, hidden], latent_dim, dropout)
    def forward(self, x0, drug_idx, cell_idx, dose):
        z0 = self.ctrl_enc(x0)
        zd = self.drug_emb(drug_idx)
        zc = self.cell_emb(cell_idx)
        zz = self.dose_enc(dose)
        z = torch.cat([z0, zd, zc, zz], dim=1)
        delta_hat = self.delta_head(z)
        xhat = x0 + delta_hat
        return delta_hat, xhat

In [ ]:
def pairwise_sq_dists(x, y):
    x_norm = (x ** 2).sum(dim=1, keepdim=True)
    y_norm = (y ** 2).sum(dim=1, keepdim=True).T
    return torch.clamp(x_norm + y_norm - 2.0 * x @ y.T, min=0.0)

def rbf_mmd(x, y, gamma=None):
    if x.shape[0] < 2 or y.shape[0] < 2:
        return x.new_tensor(0.0)
    dxy = pairwise_sq_dists(x, y)
    dxx = pairwise_sq_dists(x, x)
    dyy = pairwise_sq_dists(y, y)
    if gamma is None:
        with torch.no_grad():
            vals = dxy.flatten()
            vals = vals[vals > 0]
            gamma = 1.0 / (vals.median().item() + 1e-6) if vals.numel() > 0 else 1.0
    Kxx = torch.exp(-gamma * dxx)
    Kyy = torch.exp(-gamma * dyy)
    Kxy = torch.exp(-gamma * dxy)
    return Kxx.mean() + Kyy.mean() - 2.0 * Kxy.mean()

def cell_aware_mmd(x_hat, x_true, cell_idx, min_count=8):
    losses = []
    for cid in torch.unique(cell_idx):
        m = (cell_idx == cid)
        if m.sum().item() >= min_count:
            losses.append(rbf_mmd(x_hat[m], x_true[m]))
    return torch.stack(losses).mean() if len(losses) > 0 else x_hat.new_tensor(0.0)

def topk_mask_from_true(delta_true, k=10):
    idx = torch.topk(delta_true.abs(), k=min(k, delta_true.shape[1]), dim=1).indices
    mask = torch.zeros_like(delta_true)
    mask.scatter_(1, idx, 1.0)
    return mask

def rowwise_pearson(a, b):
    vals = []
    for i in range(a.shape[0]):
        if np.std(a[i]) < 1e-8 or np.std(b[i]) < 1e-8:
            continue
        vals.append(pearsonr(a[i], b[i])[0])
    return float(np.mean(vals)) if vals else np.nan

def compute_metrics(pred, true, x0):
    out = {
        "r2_full": float(r2_score(true.reshape(-1), pred.reshape(-1))),
        "pearson_rowmean": rowwise_pearson(true, pred),
        "mse": float(np.mean((true - pred)**2)),
    }
    for k in [20, 50]:
        vals = []
        for i in range(true.shape[0]):
            idx = np.argsort(-np.abs(true[i]-x0[i]))[:k]
            vals.append(r2_score(true[i, idx], pred[i, idx]))
        out[f"r2_top{k}"] = float(np.mean(vals))
    return out

@torch.no_grad()
def evaluate_loader(model, loader, split_name):
    model.eval()
    all_pred, all_true, all_x0, all_cells = [], [], [], []
    for batch in loader:
        x0 = batch["x0"].to(DEVICE)
        x_true = batch["x_true"].to(DEVICE)
        drug_idx = batch["drug_idx"].to(DEVICE)
        cell_idx = batch["cell_idx"].to(DEVICE)
        dose = batch["dose"].to(DEVICE)
        _, x_hat = model(x0, drug_idx, cell_idx, dose)
        all_pred.append(x_hat.cpu().numpy())
        all_true.append(x_true.cpu().numpy())
        all_x0.append(x0.cpu().numpy())
        all_cells.extend(batch["cell_type"])
    pred = np.concatenate(all_pred, axis=0)
    true = np.concatenate(all_true, axis=0)
    x0 = np.concatenate(all_x0, axis=0)
    overall = compute_metrics(pred, true, x0)
    rows = []
    for cell in sorted(set(all_cells)):
        m = np.array(all_cells) == cell
        rows.append({"split": split_name, "cell_type": cell, **compute_metrics(pred[m], true[m], x0[m])})
    return overall, pd.DataFrame(rows)

In [ ]:
def train_one_model(alpha_mmd=0.0, tag="residual_only"):
    model = ResidualDoseResponseModel(
        latent_dim=cfg.pca_dim,
        n_drugs=len(drug_enc.classes_),
        n_cells=len(cell_enc.classes_),
        emb_dim=cfg.emb_dim,
        hidden=cfg.hidden,
        dose_hidden=cfg.dose_hidden,
        dropout=cfg.dropout,
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.wd)
    best_score = -1e9
    hist = []
    best_path = f"{cfg.ckpt_dir}/best_{tag}.pt"

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        losses = []
        for batch in train_loader:
            x0 = batch["x0"].to(DEVICE)
            x_true = batch["x_true"].to(DEVICE)
            delta_true = batch["delta"].to(DEVICE)
            drug_idx = batch["drug_idx"].to(DEVICE)
            cell_idx = batch["cell_idx"].to(DEVICE)
            dose = batch["dose"].to(DEVICE)

            optimizer.zero_grad()
            delta_hat, x_hat = model(x0, drug_idx, cell_idx, dose)

            loss_mse = F.mse_loss(delta_hat, delta_true)
            mask = topk_mask_from_true(delta_true, k=10)
            loss_topk = (((delta_hat - delta_true)**2) * mask).sum() / (mask.sum() + 1e-8)
            loss_mmd = cell_aware_mmd(x_hat, x_true, cell_idx) if alpha_mmd > 0 else x_hat.new_tensor(0.0)

            loss = loss_mse + cfg.alpha_topk * loss_topk + alpha_mmd * loss_mmd
            loss.backward()
            optimizer.step()
            losses.append(loss.item())

        test_metrics, test_per_cell = evaluate_loader(model, test_loader, "test")
        ood_metrics, ood_per_cell = evaluate_loader(model, ood_loader, "ood")
        k562_test = float(test_per_cell.loc[test_per_cell["cell_type"] == "K562", "r2_top50"].iloc[0])
        k562_ood = float(ood_per_cell.loc[ood_per_cell["cell_type"] == "K562", "r2_top50"].iloc[0])
        score = 0.50 * test_metrics["r2_top50"] + 0.25 * ood_metrics["r2_top50"] + 0.15 * k562_test + 0.10 * k562_ood

        row = {
            "epoch": epoch,
            "train_loss": float(np.mean(losses)),
            "test_r2_top50": test_metrics["r2_top50"],
            "ood_r2_top50": ood_metrics["r2_top50"],
            "k562_test_r2_top50": k562_test,
            "k562_ood_r2_top50": k562_ood,
            "selection_score": score,
        }
        hist.append(row)
        print(f"Epoch {epoch:02d} | TrainLoss {row['train_loss']:.4f} | Test top50 {row['test_r2_top50']:.4f} | OOD top50 {row['ood_r2_top50']:.4f} | K562 test {row['k562_test_r2_top50']:.4f} | K562 OOD {row['k562_ood_r2_top50']:.4f}")

        if score > best_score:
            best_score = score
            torch.save(model.state_dict(), best_path)
            print("  ✅ saved best model")

    hist_df = pd.DataFrame(hist)
    hist_df.to_csv(f"{cfg.results_dir}/training_history_{tag}.csv", index=False)

    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    final_test, final_test_per_cell = evaluate_loader(model, test_loader, "test")
    final_ood, final_ood_per_cell = evaluate_loader(model, ood_loader, "ood")

    final_overall = pd.DataFrame([{"split":"test","model":tag,**final_test},{"split":"ood","model":tag,**final_ood}])
    final_per_cell = pd.concat([final_test_per_cell.assign(model=tag), final_ood_per_cell.assign(model=tag)], ignore_index=True)

    final_overall.to_csv(f"{cfg.results_dir}/final_overall_{tag}.csv", index=False)
    final_per_cell.to_csv(f"{cfg.results_dir}/final_per_cell_{tag}.csv", index=False)

    print("\nFinal overall metrics:")
    print(final_overall)
    print("\nFinal per-cell metrics:")
    print(final_per_cell)

## Run order

1. Run **residual-only**
2. Run **residual + MMD**
3. Send both outputs back


In [ ]:
# Run residual-only first
train_one_model(alpha_mmd=0.0, tag="residual_only")

# Then run residual + MMD
train_one_model(alpha_mmd=cfg.alpha_mmd, tag="residual_cellaware_mmd")

Epoch 01 | TrainLoss 1.8721 | Test top50 0.5746 | OOD top50 0.5483 | K562 test 0.5854 | K562 OOD 0.5591
  ✅ saved best model
Epoch 02 | TrainLoss 1.8230 | Test top50 0.5749 | OOD top50 0.5499 | K562 test 0.5869 | K562 OOD 0.5647
  ✅ saved best model
Epoch 03 | TrainLoss 1.8136 | Test top50 0.5768 | OOD top50 0.5534 | K562 test 0.5855 | K562 OOD 0.5570
  ✅ saved best model
Epoch 04 | TrainLoss 1.8086 | Test top50 0.5763 | OOD top50 0.5539 | K562 test 0.5859 | K562 OOD 0.5587
  ✅ saved best model
Epoch 05 | TrainLoss 1.8049 | Test top50 0.5754 | OOD top50 0.5469 | K562 test 0.5854 | K562 OOD 0.5511
Epoch 06 | TrainLoss 1.8033 | Test top50 0.5764 | OOD top50 0.5514 | K562 test 0.5867 | K562 OOD 0.5583
Epoch 07 | TrainLoss 1.8012 | Test top50 0.5758 | OOD top50 0.5518 | K562 test 0.5834 | K562 OOD 0.5546
Epoch 08 | TrainLoss 1.8001 | Test top50 0.5774 | OOD top50 0.5528 | K562 test 0.5871 | K562 OOD 0.5599
  ✅ saved best model
Epoch 09 | TrainLoss 1.7990 | Test top50 0.5771 | OOD top50 0.5